In [ ]:
import sys
from pathlib import Path
from pickle import dump, load

try:
    _here = Path(__vsc_ipynb_file__).parent  # VS Code
except NameError:
    _here = Path().resolve() / 'demo'  # JupyterLab (CWD = project root)
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import anndata as ad
import numpy as np
import pandas as pd
from ingest import remyelin_ds, run_nmf, run_proc_per_ds
from plotnine import *  # noqa: F403

import apl
import spatialpeeler as sP
from spatialpeeler.diag import comp_sparsity, geom_point_sp, run_deg, sweep_uns

In [ ]:
RAND_SEED = 28

In [ ]:
# load dataset + run NMF
if (pth := Path("/lustre/scratch126/gengen/teams_v2/marks/dp31/sP_demo_data_dir/.cache/rml.pkl")).exists():
    with open(pth, "rb") as f:
        rml = load(f)
else:
    adatas = remyelin_ds(
        Path("/lustre/scratch126/gengen/teams_v2/marks/dp31/sP_demo_data_dir/remyelin-slideseq"),
        # load only subset of interest
        lambda obs: (
            (obs["GoodQuality"] == "Yes")
            & (
                ((obs["Timepoint"].isin([3])) & (obs["Condition"] == "LPC"))
                | (obs["Timepoint"].isin([12, 18]) & (obs["Condition"] == "Saline"))
            )
        ),
    )
    run_proc_per_ds(adatas)

    rml = ad.concat(adatas, label="puck_id", index_unique="_")
    rml.obs["puck_id_cond"] = rml.obs["puck_id"].str.cat(rml.obs["Condition"], sep="_")

    run_nmf(rml, 30, RAND_SEED)

    with open(pth, "wb") as f:
        dump(rml, f)
print(rml)

before proceeding with spatialPeeler, we inspect our count matrix and NMF factors to assess their sparsity

In [ ]:
X_spars = rml.X.nnz / np.prod(rml.X.shape)  # ty:ignore[unresolved-attribute]
(
    comp_sparsity(rml.obsm["X_nmf"], cond=rml.obs["Condition"])
    >> ggplot()
    + aes(x="column", colour="cond", y="sparsity")
    + geom_point()
    + labs(x="NMF factor", title=f"count sparsity: {X_spars * 100:.02f}%")
).show()

we see that the dataset is overall quite sparse (only ~1% of the data is non zero)
and this is reflected in the factors, which are also mostly quite sparse

as such, we threshold our factor regression analysis to only include cells which are non-sparse for that factor

In [ ]:
# run factor-wise regression step
for k in range(rml.obsm["X_nmf"].shape[1]):
    sP.run_fact_reg(
        rml,
        "Condition",
        "Saline",
        k,
        f"sp_p_hat_k={k:02}",
        X_key="X_nmf",
        filt=lambda adata: adata[adata.obsm["X_nmf"][:, k] > 0, :],
    )

print(rml)

we now assess our resulting regression outputs distributions
we can extract and collate the model betas and other statistics using the sweep_uns function

In [ ]:
(
    sweep_uns(rml, "sp_p_hat_k={k}")
    >> ggplot()
    + aes(x="reorder(k, -beta)", fill="-np.log(pval)", y="beta")
    + geom_col()
    + labs(fill="-log(pval)", x="NMF factor")
    + scale_fill_cmap("turbo")
).show()

we can see that factor 6 has particularly high beta value and significance,
so we can plot the spatial distribution of the factor and the pattern

In [ ]:
rml_filt = rml[rml.obsm["X_nmf"][:, 6] > 0, :]
k_oi = "sp_p_hat_k=06"
plo = apl.PlotAdata(rml_filt)
pl1a = plo + aes(colour=apl.obsm("X_nmf", 6)) + labs(colour="NMF[6]") + scale_colour_cmap("YlGnBu")
pl1b = plo + aes(colour=apl.obsm(k_oi, sP.PAT_KEY)) + labs(colour="p_hat") + scale_colour_cmap("YlGnBu", limits=(0, 1))
pl1 = (
    (pl1a / pl1b)
    & aes(x=apl.obsm("spatial", "x"), y=apl.obsm("spatial", "y"))
    & geom_point_sp(size_in_px=2, alpha=0.5)
    & facet_wrap(apl.obs("puck_id_cond"))
    & theme(
        axis_text=element_blank(),
        axis_ticks=element_blank(),
        strip_text=element_blank(),
    )
)
pl2 = (
    plo
    + aes(
        x=apl.obsm("X_nmf", 6),
        y=apl.obsm(k_oi, sP.PAT_KEY),
        colour=apl.obs("Condition"),
    )
    + geom_point(alpha=0.5, shape=".")
    + labs(y="p_hat")
    + scale_colour_discrete()
)
(pl1 | pl2).show()

In [ ]:
# NOTE (known bug in apl.PlotAdata): each `+` on a PlotAdata calls ggplot.__deepcopy__,
# which iterates __dict__ and deep-copies _PlotAdata__adata (the name-mangled __adata field).
# PlotAdata.__deepcopy__ tries to prevent this by reassigning __adata *after* super().__deepcopy__,
# but the full AnnData has already been copied by then — so every `+` is very slow on large datasets.
# Workaround: extract what's needed into a plain DataFrame and use ggplot directly.

_df = pd.DataFrame({
    "x": rml_filt.obsm["spatial"]["x"].values,
    "y": rml_filt.obsm["spatial"]["y"].values,
    "NMF_6": rml_filt.obsm["X_nmf"][:, 6],
    "p_hat": np.asarray(rml_filt.obsm[k_oi][sP.PAT_KEY]),
    "Condition": rml_filt.obs["Condition"].values,
    "puck_id_cond": rml_filt.obs["puck_id_cond"].values,
})

_pl1a = (
    ggplot(_df, aes(x="x", y="y", colour="NMF_6"))
    + geom_point_sp(size_in_px=2, alpha=0.5)
    + facet_wrap("puck_id_cond")
    + scale_colour_cmap("YlGnBu")
    + labs(colour="NMF[6]")
    + theme(axis_text=element_blank(), axis_ticks=element_blank(), strip_text=element_blank())
)
_pl1b = (
    ggplot(_df, aes(x="x", y="y", colour="p_hat"))
    + geom_point_sp(size_in_px=2, alpha=0.5)
    + facet_wrap("puck_id_cond")
    + scale_colour_cmap("YlGnBu", limits=(0, 1))
    + labs(colour="p_hat")
    + theme(axis_text=element_blank(), axis_ticks=element_blank(), strip_text=element_blank())
)
_pl2 = (
    ggplot(_df, aes(x="NMF_6", y="p_hat", colour="Condition"))
    + geom_point(alpha=0.5, shape=".")
    + labs(y="p_hat")
    + scale_colour_discrete()
)
((_pl1a / _pl1b) | _pl2).show()

we can observe that the non-linearity of the logistic regression probability outputs allows
for strong extraction of the disease effect, which is not as visible with just the NMF factor on its own

we can also use the run_deg function provided in the sP.diag module to assess DEGs between various labels / combinations of labels

In [ ]:
deg_results = {
    "case1_vs_other": run_deg(rml_filt, {apl.obsm(k_oi, sP.BIN_KEY): True, apl.obs("Condition"): "LPC"}),
    "case1_vs_case0": run_deg(rml_filt[rml_filt.obs["Condition"] == "LPC"], {apl.obsm(k_oi, sP.BIN_KEY): True}),
    "case1_vs_control1": run_deg(
        rml_filt[np.asarray(rml_filt.obsm[k_oi][sP.BIN_KEY])],
        {apl.obs("Condition"): "LPC"},
    ),
    "case0_vs_control": run_deg(
        rml_filt,
        {apl.obsm(k_oi, sP.BIN_KEY): False, apl.obs("Condition"): "LPC"},
        {apl.obs("Condition"): "Saline"},
    ),
    "case0_vs_control0": run_deg(
        rml_filt[~np.asarray(rml_filt.obsm[k_oi][sP.BIN_KEY])],
        {apl.obs("Condition"): "LPC"},
    ),
    "control1_vs_control0": run_deg(
        rml_filt[rml_filt.obs["Condition"] == "Saline"],
        {apl.obsm(k_oi, sP.BIN_KEY): True},
    ),
}

In [ ]:
(
    pd.concat(deg_results, names=["cmp"])
    .reset_index(drop=False)
    .loc[lambda e: e["adj_pval"] < 0.05]
    .groupby("cmp", observed=True)["gene"]
    .count()
    .reset_index(drop=False)
    >> ggplot()
    + aes(x="cmp", y="gene", fill=after_stat("y"))
    + geom_col()
    + theme(axis_text_x=element_text(angle=45, hjust=1))
    + guides(fill="none")
    + labs(x="", y="number of significant DEGs")
)

we can also assess the distributions of p_hat values across the binarized labels:

In [ ]:
(
    plo
    + aes(
        x=apl.obs("Condition"),
        y=apl.obsm(k_oi, sP.PAT_KEY),
        fill=apl.obsm(k_oi, sP.BIN_KEY),
    )
    + geom_violin()
    + labs(fill="binarized\nlabel")
    + lims(y=(0, 1))
).show()

given factor 6 displays interesting properties, we proceed with gene-wise analysis for factor 6

In [ ]:
ks_oi = [6]
for k in ks_oi:
    sP.run_gene_reg(rml, f"sp_p_hat_k={k:02}", f"sp_gene_id_k={k:02}", layer="counts")


(
    apl.PlotAdata(rml)
    + aes(
        x=apl.varm("X_nmf", 6),
        y=apl.varm("sp_gene_id_k=06", "reg_beta"),
        colour=stage(apl.varm("sp_gene_id_k=06", "reg_pval"), "-np.log(color)"),
    )
    + geom_point(size=1.5, shape="o", stroke=0)
    + labs(colour="-log(pval)", y="model beta of NMF[6] w/ p_hat")
)